# nb_mig_silver_20261116100000_SeedRegSub_TransformMappings_Unified_B

Create the metadata entried for the various silver unified tables for Registration Submissions. 

Note, since multiple developers are working on this simultaneously, the metadata insertion has been divided across multiple migration notebooks

In [1]:
%run "/05 - shared-libraries/nb_shared_common"

In [2]:
%run "/05 - shared-libraries/nb_shared_metadata"

In [4]:
get_max_transform_order(
    "metadata2_db",
    "silver_master_unify"
)

In [3]:
%run "/05 - shared-libraries/nb_shared_migrations"

In [4]:
storage_account_name = "deintdlsaueaasqebla"
dataverse_container = "dataverse-asqaarms01de-unqde396ad0f22d448c94c779e588420"

In [5]:
#imports
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime, timezone
from delta.tables import DeltaTable
from mdw_shared.migrations import ensure_history_table, is_applied, record_migration

# Get spark session and delta lake paths
spark = get_spark_session()

data_lake_paths = get_datalake_paths(storage_account_name, dataverse_container)
METADATA_BASE_PATH = data_lake_paths["metadata"]

VERSION_ID = "v1"

MIGRATION_ID = "20261116100000_SeedRegSub_TransformMappings_Unified_B"
NOTEBOOK_NAME = "nb_mig_silver_20261116100000_SeedRegSub_TransformMappings_Unified_B"

ensure_history_table(spark, METADATA_BASE_PATH)


In [9]:

if is_applied(spark, METADATA_BASE_PATH, MIGRATION_ID):
    print(f"Already applied: {MIGRATION_ID}")
else:
    now = datetime.now(timezone.utc)

    is_active = True
    created_utc = now
    updated_utc = now

    # =================================================================
    # PART 1: Transform Mappings
    # =================================================================

    NOTEBOOK_TO_EXECUTE_NONE = None
    JSON_CONFIG_NONE = None
    SOURCE_ID_NONE = None
    SOURCE_ENTITY_NONE = None

    # The order is no longer typed in by hand. Rows below carry NO order value; assign_orders()
    # numbers them sequentially per step, continuing from the highest order already in the metadata
    # table when this migration runs. That keeps the numbering unique and gap-free no matter how many
    # migration notebooks are adding rows, and nobody needs to know what the other notebooks used.
    # NOTE: this only guarantees unique, sequential numbers. Within each list, put parents before
    # children (the order here is the run order).

    def assign_orders(rows, start_by_step):
        counters = dict(start_by_step)
        out = []
        for binding, step, source_id, source_entity, target_entity, notebook, json_file in rows:
            counters[step] += 1
            out.append((binding, step, source_id, counters[step], source_entity, target_entity, notebook, json_file))
        return out

    # Read the current max at execution time, not at notebook-write time
    start_by_step = {
        STEP_SILVER_MASTER_UNIFY: get_max_transform_order("metadata2_db", STEP_SILVER_MASTER_UNIFY) or 0,
        STEP_SILVER_TX_UNIFY:     get_max_transform_order("metadata2_db", STEP_SILVER_TX_UNIFY) or 0,
    }

    # transform_entity_mappings contains the metadata entries to map ARMS to Clease
    # and cleanse to unify
    # Parameter 1 : Binding Id
    # Parameter 2 : Step Name
    # Parameter 3 : Source Id (only relevant for ARMS cleanse, for unifcation this is set to SOURCE_ID_NONE)
    # Parameter 4 : Source entity name
    # Parameter 5 : Target entity name
    # Parameter 6 : Any custom notebook to run for the transformation only if relevant
    # Parameter 7 : name of JSON mapping file to map fields and run relevant validations at a field level
    # (Order is added by assign_orders)

    master_rows = [
         # Unify Master
        (BINDING_SILVER_MASTER_UNIFY_ID, STEP_SILVER_MASTER_UNIFY, SOURCE_ID_NONE, "asqa_governingpersonrole", "governing_person_role", NOTEBOOK_TO_EXECUTE_NONE, "governing_person_role.json"),
        (BINDING_SILVER_MASTER_UNIFY_ID, STEP_SILVER_MASTER_UNIFY, SOURCE_ID_NONE, "asqa_rtotypeofsignificantevent", "rto_type_of_significant_event", NOTEBOOK_TO_EXECUTE_NONE, "rto_type_of_significant_event.json"),
        (BINDING_SILVER_MASTER_UNIFY_ID, STEP_SILVER_MASTER_UNIFY, SOURCE_ID_NONE, "asqa_studentmanagementprovider", "student_management_provider", NOTEBOOK_TO_EXECUTE_NONE, "student_management_provider.json"),
    ]

    tx_rows = [
         # Unify Transaction
         # asqa_registrationsubmission is the parent of the reg_sub_* tables below, so it goes first
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_asqainvoice", "invoice", NOTEBOOK_TO_EXECUTE_NONE, "invoice.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rfi", "reg_sub_rfi", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rfi.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtodelegation", "reg_sub_rto_delegation", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_delegation.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtoevidenceofdelivery", "reg_sub_rto_evidence_of_delivery", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_evidence_of_delivery.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtofinancialstatus1", "reg_sub_rto_financial_status", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_financial_status.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtolegaldetails", "reg_sub_rto_legal_detail", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_legal_detail.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_rtoownership", "reg_sub_rto_ownership", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_rto_ownership.json"),
        (BINDING_SILVER_TX_UNIFY_ID, STEP_SILVER_TX_UNIFY, SOURCE_ID_NONE, "asqa_thirdpartyarrangements", "reg_sub_third_party_arrangement", NOTEBOOK_TO_EXECUTE_NONE, "reg_sub_third_party_arrangement.json"),
    ]

    transform_entity_mappings = assign_orders(master_rows, start_by_step) + assign_orders(tx_rows, start_by_step)

    for m in transform_entity_mappings:
        print(m[1], m[3], m[4], "->", m[5])

    insert_transform_entity_mappings(METADATA_BASE_PATH, transform_entity_mappings, VERSION_ID, spark)

    # TODO: remove before PR
    #record_migration(spark, METADATA_BASE_PATH, MIGRATION_ID, BINDING_STAGE_SILVER, NOTEBOOK_NAME, "succeeded")
    print(f"Applied: {MIGRATION_ID}")

In [8]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col
table_name = "metadata2_db.transform_businessready_entity_mapping"
source_entities_to_delete = [
    "asqa_governingpersonrole",
    "asqa_rtotypeofsignificantevent",
    "asqa_studentmanagementprovider",
    "asqa_asqainvoice",
   # "asqa_registrationsubmission",
    "asqa_rfi",
    "asqa_rtodelegation",
    "asqa_rtoevidenceofdelivery",
    "asqa_rtofinancialstatus1",
    "asqa_rtolegaldetails",
    "asqa_rtoownership",
    "asqa_thirdpartyarrangements"
]
# Set to True only after the dry run output looks right
RUN_DELETE = True #False
cond = col("source_entity_name").isin(source_entities_to_delete)
# 1. Dry run: what will be deleted
before_total = spark.table(table_name).count()
to_delete = spark.table(table_name).filter(cond)
to_delete_count = to_delete.count()
print(f"Rows in table: {before_total}")
print(f"Rows to delete: {to_delete_count}")
to_delete.groupBy("source_entity_name").count().orderBy("source_entity_name").show(50, False)
# 2. Restore point
version_before = DeltaTable.forName(spark, table_name).history(1).collect()[0]["version"]
print(f"Current version (restore point): {version_before}")
# 3. Delete only the matching rows, then verify
if RUN_DELETE:
    DeltaTable.forName(spark, table_name).delete(cond)
    after_total = spark.table(table_name).count()
    remaining = spark.table(table_name).filter(cond).count()
    print(f"Rows after: {after_total} (expected {before_total - to_delete_count})")
    print(f"Matching rows remaining: {remaining} (expected 0)")
    # Undo, if needed:
    # spark.sql(f"RESTORE TABLE {table_name} TO VERSION AS OF {version_before}")
else:
    print("Dry run only. Set RUN_DELETE = True to delete.")  